In [1]:
import os
import pydicom
import pandas as pd

In [2]:
# Configuration
# Go from analysis folder to data folder
root_directory = os.path.join('..', 'data', 'PANCREAS_2', 'PANCREAS_2')

reports_dir = os.path.join('reports')
os.makedirs(reports_dir, exist_ok=True)
output_filename = os.path.join(reports_dir, 'dataset_audit_simple.csv')

# List to hold the data for every image we find
image_data_list = []

print(f"Scanning directory: {os.path.abspath(root_directory)}")
print(f"Reports directory: {os.path.abspath(reports_dir)}")

Scanning directory: /home/daniduhnev/projects/master-thesis/data/PANCREAS_2/PANCREAS_2
Reports directory: /home/daniduhnev/projects/master-thesis/analysis/reports


In [3]:
# Walk through all folders
for current_folder, _, files in os.walk(root_directory):
    
    for filename in files:
        if filename.startswith("."): 
            continue
            
        full_path = os.path.join(current_folder, filename)
        
        try:
            # Read header only
            ds = pydicom.dcmread(full_path, stop_before_pixels=True)
            
            image_data_list.append({
                "filename": filename,
                "height": ds.get("Rows", 0),
                "width": ds.get("Columns", 0),
                "channels": ds.get("SamplesPerPixel", 1),
                "modality": ds.get("Modality", "Unknown"),
                "full_path": full_path
            })
            
        except Exception as e:
            print(f"Warning: Could not read file {filename}: {e}")

In [4]:
# Convert our list to a pandas DataFrame (like an Excel sheet)
df_images = pd.DataFrame(image_data_list)

In [5]:
print("Dataset Audit Report\n")
print(f"Total Images: {len(df_images)}")

print("\nResolutions (Height x Width):")
print(df_images.groupby(['height', 'width']).size())

print("\nChannels (1=Gray, 3=RGB):")
print(df_images['channels'].value_counts())

print("\nModalities:")
print(df_images['modality'].value_counts())

df_images.to_csv(output_filename, index=False)
print(f"\nData saved to: {output_filename}")

Dataset Audit Report

Total Images: 134

Resolutions (Height x Width):
height  width
768     1024     134
dtype: int64

Channels (1=Gray, 3=RGB):
channels
3    134
Name: count, dtype: int64

Modalities:
modality
US    134
Name: count, dtype: int64

Data saved to: reports/dataset_audit_simple.csv


In [ ]:
print(f"Scanning for duplicates in: {os.path.abspath(root_directory)}\n")

duplicates = []

for current_folder, _, files in os.walk(root_directory):
    valid_files = [f for f in files if not f.startswith(".")]
    if len(valid_files) > 1:
        duplicates.append({
            "folder": current_folder,
            "files": valid_files
        })

print("\n")
if duplicates:
    print(f"Found {len(duplicates)} folders with multiple images:\n")
    for item in duplicates:
        print(f"Folder: {os.path.basename(item['folder'])}")
        print(f"Path:   {item['folder']}")
        print(f"Count:  {len(item['files'])} images")
        print("Files:")
        for f in item['files']:
            print(f"  - {f}")
else:
    print("No duplicates found. Each folder contains exactly 1 image.")

Scanning for duplicates in: /home/daniduhnev/projects/master-thesis/data/PANCREAS_2/PANCREAS_2

Duplicate Scan Report

No duplicates found. Each folder contains exactly 1 image.
